# Modelo

In [1]:
import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED

Elegimos dataset

In [2]:
dataset = 'ciclos_f2_r'

## Espectrogramas

Se suelen usar Mel espectrogramas

Traigo los consjuntos de entrenamiento y testeo. (TODO: buscar una manera más eficiente de guardarlos)

In [3]:
train_data = np.load(f'./dataset/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2642, 23808), (2642,), (303, 23808), (303,))

### Random Forest

#### Entrenamiento

In [5]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(8, 20),
    'min_samples_split': randint(5, 25),
    'min_samples_leaf': randint(5, 25)
}

In [6]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='recall',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=14, min_samples_leaf=24, min_samples_split=19;, score=0.533 total time=   2.1s
[CV 2/5] END max_depth=14, min_samples_leaf=24, min_samples_split=19;, score=0.566 total time=   1.9s
[CV 3/5] END max_depth=14, min_samples_leaf=24, min_samples_split=19;, score=0.577 total time=   1.8s
[CV 4/5] END max_depth=14, min_samples_leaf=24, min_samples_split=19;, score=0.593 total time=   1.8s
[CV 5/5] END max_depth=14, min_samples_leaf=24, min_samples_split=19;, score=0.632 total time=   1.7s
[CV 1/5] END max_depth=18, min_samples_leaf=12, min_samples_split=11;, score=0.545 total time=   2.0s
[CV 2/5] END max_depth=18, min_samples_leaf=12, min_samples_split=11;, score=0.620 total time=   2.4s
[CV 3/5] END max_depth=18, min_samples_leaf=12, min_samples_split=11;, score=0.627 total time=   3.2s
[CV 4/5] END max_depth=18, min_samples_leaf=12, min_samples_split=11;, score=0.593 total time=   3.4s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....001F83535EA50>, 'min_samples_leaf': <scipy.stats....001F8353B4190>, 'min_samples_split': <scipy.stats....001F8028B39D0>}"
,n_iter,20
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(20)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
4,15,7,6,0.625016
6,9,5,16,0.624186
10,16,7,9,0.615912
19,11,6,10,0.612602
3,18,8,12,0.605130
1,18,12,11,0.602661
16,19,12,19,0.601835
8,18,14,20,0.596859
5,15,16,10,0.595202
9,10,16,24,0.594393


In [8]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 15, 'min_samples_leaf': 7, 'min_samples_split': 6}
Best CV score: 0.6250162888789822


In [9]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1434
           1       0.99      0.97      0.98      1208

    accuracy                           0.98      2642
   macro avg       0.98      0.98      0.98      2642
weighted avg       0.98      0.98      0.98      2642



#### Evaluación

In [10]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.86      0.80       180
           1       0.74      0.58      0.65       123

    accuracy                           0.75       303
   macro avg       0.74      0.72      0.72       303
weighted avg       0.75      0.75      0.74       303



#### Guardado

In [11]:
joblib.dump(best_model, f'./modelos/{dataset}/melspec_rf.pkl')

['./modelos/ciclos_f2_r/melspec_rf.pkl']

## Atributos de Audio

1. Resample y Filtro pasa bajos
2. Feature Extractor
    + MFCC (13)
    + ZCR
    + Short-Time Energy
    + SC
    + Spectral Roll-off
    + BER
    + Spectral Flatness

In [12]:
train_data = np.load(f'./dataset/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [13]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2642, 19), (2642,), (303, 19), (303,))

### Random Forest

#### Entrenamiento

In [14]:
rf = RandomForestClassifier(random_state=42, max_features=None, n_jobs=-1)

param_distributions = {
    'max_depth': randint(5, 20),
    'min_samples_split': randint(5, 25),
    'min_samples_leaf': randint(5, 25)
}

In [15]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='recall',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.504 total time=   0.4s
[CV 2/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.550 total time=   0.3s
[CV 3/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.519 total time=   0.3s
[CV 4/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.614 total time=   0.4s
[CV 5/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.570 total time=   0.3s
[CV 1/5] END max_depth=15, min_samples_leaf=12, min_samples_split=11;, score=0.574 total time=   0.4s
[CV 2/5] END max_depth=15, min_samples_leaf=12, min_samples_split=11;, score=0.583 total time=   0.4s
[CV 3/5] END max_depth=15, min_samples_leaf=12, min_samples_split=11;, score=0.573 total time=   0.4s
[CV 4/5] END max_depth=15, min_samples_leaf=12, min_samples_split=11;, score=0.676 total time=   0.4s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....001F8352CF820>, 'min_samples_leaf': <scipy.stats....001F8353429F0>, 'min_samples_split': <scipy.stats....001F8352CE8B0>}"
,n_iter,50
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [16]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
40,18,7,5,0.639923
3,15,8,12,0.635801
29,12,5,16,0.626679
4,12,7,6,0.621697
34,16,11,13,0.616742
33,14,11,13,0.616735
48,17,11,23,0.615908
1,15,12,11,0.615082
42,15,13,19,0.612596
16,16,12,19,0.612596


In [17]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 18, 'min_samples_leaf': 7, 'min_samples_split': 5}
Best CV score: 0.6399231850759577


In [18]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      1434
           1       0.94      0.92      0.93      1208

    accuracy                           0.94      2642
   macro avg       0.94      0.94      0.94      2642
weighted avg       0.94      0.94      0.94      2642



#### Evaluación

In [19]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.88      0.83       180
           1       0.79      0.65      0.71       123

    accuracy                           0.79       303
   macro avg       0.79      0.77      0.77       303
weighted avg       0.79      0.79      0.78       303



#### Guardado

In [20]:
joblib.dump(best_model, f'./modelos/{dataset}/features_rf.pkl')

['./modelos/ciclos_f2_r/features_rf.pkl']